In [ ]:
from dlfs.layers import DenseLayer
from dlfs.activation import ReLU, Sigmoid, Softmax
from dlfs.loss import BCE_Loss, CCE_Loss
from dlfs.optimizers import Optimizer_SGD, Optimizer_Adam
from dlfs.model import SequentialModel

from sklearn.datasets import make_circles, make_moons, make_blobs

from viz_helpers import *

# Dense Layer visualisation for classification

- this notebook aims to visualize 1D, 2D, 3D linear transformations (by Dense Layers) and activation functions applied to classification data

# Different datasets and layers for visualisation

In [ ]:
def make_xor():

    X = np.array([[0, 0],
                  [0, 1],
                  [1, 0],
                  [1, 1]])
    y = np.array([0, 1, 1, 0])

    return X, y

def init_layers(dataset):

    circles_layers = [
        DenseLayer(2, 3),
        ReLU(),
        DenseLayer(3, 1), # Output is 1 class
        Sigmoid()
    ]

    xor_layers = [
        DenseLayer(2, 2),
        Sigmoid(),
        DenseLayer(2, 1), # Output is 1 class
        Sigmoid()
    ]

    moons_layers =  [
        DenseLayer(2, 3), 
        ReLU(),
        DenseLayer(3, 3),
        ReLU(),
        DenseLayer(3, 1), # Output is 1 class
        Sigmoid()
    ]

    blobs_layers = [
        DenseLayer(2, 3), 
        ReLU(),
        DenseLayer(3, 3),
        ReLU(),
        DenseLayer(3, 3), # Output is 3 classes
        Softmax()
    ]

    match dataset:
        case "circles":
            return circles_layers
        case "moons":
            return moons_layers
        case "xor":
            return xor_layers
        case "blobs":
            return blobs_layers
        case _:
            return []

def get_dataset_loss_fn(dataset):
    if dataset != "blobs":
        return BCE_Loss()
    else:
        return CCE_Loss()

datasets = {
    "circles": lambda: make_circles(n_samples=300, factor=0.4, noise=0.1),
    "moons": lambda: make_moons(n_samples=200, noise=0.1),
    "xor": make_xor,
    "blobs": lambda: make_blobs(n_samples=300, n_features=2, centers=3, cluster_std=1.2),
}

# Linearly nonseparable dataset

In [ ]:
dataset = "moons"
X, y = datasets[dataset]()
plot_2d_clf_problem(X, y)

# Training classification model

In [ ]:
layers = init_layers(dataset)
loss_fn = get_dataset_loss_fn(dataset)
lr = 1e-2
epochs = 5000

model = SequentialModel(
    layers=layers, 
    loss_function=loss_fn, 
    optimizer=Optimizer_SGD(learning_rate=lr)
)

if dataset != "blobs":
    y_train = y.reshape(-1, 1)
else:
    y_train = y

model.train(X, y_train, print_every=1000, epochs=epochs)

# Show training result

In [ ]:
if dataset != "blobs":
    y_pred = np.round(model.predict(X))
    acc = np.mean(y_train == y_pred)
    print(f'Accuracy: {acc:.4f}')
    plot_2d_clf_problem(X, y, lambda x: model.predict(x) > 0.5)
else:
    y_pred = np.argmax(model.predict(X), axis=-1)
    acc = np.mean(y_train == y_pred)
    print(f'Accuracy: {acc:.4f}')
    plot_2d_clf_problem(X, y, lambda x: np.argmax(model.predict(x)))


# Extract each layer's output

In [ ]:
model.forward(X)

Z1 = model.wrapper.layers[0].output.copy() # first dense layer
A1 = model.wrapper.layers[1].output.copy() # first dense + activation
Z2 = model.wrapper.layers[2].output.copy() # second dense
A2 = model.wrapper.layers[3].output.copy() # second dense + activation

layer1_dim = Z1.shape[1]
layer2_dim = Z2.shape[1]
layer3_dim = None

if dataset in ["moons", "blobs"]:
    Z3 = model.wrapper.layers[4].output.copy() # third dense
    A3 = model.wrapper.layers[5].output.copy() # third dense + activation
    layer3_dim = Z3.shape[1]

# Plotting first linear transformation

In [ ]:
title = f"2D → {layer1_dim}D using first Dense Layer"

if layer1_dim == 2:
    plot_2d_classification_output(Z1, y, title)
elif layer1_dim == 3:
    plot_3d_classification_output(Z1, y, title)

# Plotting first linear transformation and first activation

In [ ]:
title = f"2D → {layer1_dim}D using first Dense Layer + Activation"

if layer1_dim == 2:
    plot_2d_classification_output(A1, y, title)
elif layer1_dim == 3:
    plot_3d_classification_output(A1, y, title)

# Plotting second linear transformation

In [ ]:
title = f"{layer1_dim}D → {layer2_dim}D using second Dense Layer"

if layer2_dim == 3:
    plot_3d_classification_output(Z2, y, title)
elif layer2_dim == 2:
    plot_2d_classification_output(Z2, y, title)
elif layer2_dim == 1:
    plot_1d_classification_output(Z2, y, title, logits=True)


# Plotting second linear transformation and second activation

In [ ]:
title = f"{layer1_dim}D → {layer2_dim}D using second Dense Layer + Activation"

if layer2_dim == 3:
    plot_3d_classification_output(A2, y, title)
elif layer2_dim == 2:
    plot_2d_classification_output(A2, y, title)
elif layer2_dim == 1:
    plot_1d_classification_output(A2, y, title)


In [ ]:
if layer3_dim is not None:
    title = f"{layer2_dim}D → {layer3_dim}D using third Dense Layer"

    if layer3_dim == 1:
        plot_1d_classification_output(Z3, y, title, logits=True)
    elif layer3_dim == 3:
        plot_3d_classification_output(Z3, y, title)


In [ ]:
if layer3_dim is not None:
    title = f"{layer2_dim}D → {layer3_dim}D using third Dense Layer + Activation"

    if layer3_dim == 1:
        plot_1d_classification_output(A3, y, title)
    elif layer3_dim == 3:
        plot_3d_classification_output(A3, y, title)


In [ ]:
if layer3_dim is not None:
    title = f"{layer2_dim}D → {layer3_dim}D using third Dense Layer + Activation + Round"
    if layer3_dim == 3:
        plot_3d_classification_output(np.round(A3), y, title)